In [ ]:
# Install Mazinger from Local Kaggle Dataset
import os
import subprocess
mazinger_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "pyproject.toml" in files and "mazinger" in root.lower():
        mazinger_path = root
        break
if mazinger_path:
    print(f"Found local mazinger at {mazinger_path}, installing...")
    # Use --no-deps to avoid conflicting versions of numpy/numba on kaggle which causes errors
    subprocess.run(["pip", "install", "--no-deps", "--no-cache-dir", f"{mazinger_path}[all]"], check=True)
    # Install required dependencies explicitly that are not preinstalled on Kaggle
    subprocess.run(["pip", "install", "--no-cache-dir", "faster-whisper", "deepgram-sdk", "demucs", "omnivoice"], check=True)
else:
    print("Local mazinger not found! Failing back to git (requires internet).")
    subprocess.run(["pip", "install", "--no-cache-dir", "git+https://github.com/bakrianoo/mazinger.git#egg=mazinger[all]"], check=True)


In [ ]:
# Task 2: Find the uploaded video in Kaggle input directory
import os
import glob
import shutil
import json

input_dir = "/kaggle/input"
video_path = None

for root, dirs, files in os.walk(input_dir):
    for file in files:
        if file.endswith(('.mp4', '.mkv', '.avi', '.mov')):
            video_path = os.path.join(root, file)
            break
    if video_path:
        break

if not video_path:
    raise FileNotFoundError("Video not found in Kaggle input directory!")
print(f"Found video at: {video_path}")

In [ ]:
# Execute Mazinger Pipeline wrapped in Try-Except
import traceback
import subprocess

try:
    print("\U0001f680 Starting Mazinger Dubbing Pipeline...")
    base_dir = "/kaggle/working/mazinger_output"
    os.makedirs(base_dir, exist_ok=True)
    
    # Execute Mazinger CLI
    import sys
    cmd = [
        sys.executable, "-m", "mazinger", "dub", video_path, 
        "--target-language", "Hindi", 
        "--output-type", "video",
        "--transcribe-method", "faster-whisper" # Free, no API keys required
    ]
    
    print(f"Executing: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=base_dir, capture_output=True, text=True)
    
    print(f"STDOUT:\n{result.stdout}")
    print(f"STDERR:\n{result.stderr}")
    
    if result.returncode != 0:
        raise Exception(f"Mazinger CLI failed with return code {result.returncode}")
    
    print("\u2705 Mazinger finished successfully.")
    
    # Copy final video and report to /kaggle/working/
    output_candidates = []
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.lower() == 'dubbed.mp4':
                output_candidates.append(os.path.join(root, file))
    output_video = output_candidates[-1] if output_candidates else None
    
    if output_video:
        shutil.copy(output_video, "/kaggle/working/output.mp4")
        print("Copied final video to /kaggle/working/output.mp4")
    else:
        err_msg = "\u26a0\ufe0f Could not find output.mp4 from Mazinger!"
        print(err_msg)
        with open("/kaggle/working/error_log.txt", "w") as f:
            f.write(err_msg + "\n")
        
    # Generate report.json
    report_data = {
        "Status": "Success" if output_video else "Failed",
        "Target_Language": "Hindi",
        "Output_File": os.path.basename(output_video) if output_video else "Missing",
        "WER": "Not measured",
        "Sync_Offset": "Not measured",
        "Speaker_Similarity": "Not measured"
    }
    with open("/kaggle/working/report.json", "w") as f:
        json.dump(report_data, f)
        
except Exception as e:
    error_text = traceback.format_exc()
    print(f"\u274c Error occurred!\n{error_text}")
    with open("/kaggle/working/error_log.txt", "w") as f:
        f.write(error_text)
